# 강사용 — 로봇 제어 실습

학생이 쓰는 명령은 `Arm`과 `load_pick_targets`뿐입니다. 이 노트북은 그 명령을 가상 팔로 보여주는 순서입니다.

프로젝트 폴더에서 엽니다. 커널은 `lerobot`입니다. 셀은 위에서 아래로 한 번씩 실행합니다. `Arm()` 셀을 두 번 켜면 팔이 겹칩니다.

## 수업 전

- 펜던트(`python pendant/main.py`)는 끕니다. 이 노트북이 Meshcat을 따로 띄웁니다.
- 브라우저가 안 열리면 출력된 Meshcat 주소를 직접 엽니다.
- 좌표는 로봇 베이스입니다. **+X 앞, +Y 왼쪽, +Z 위.** 위치는 mm, 자세는 도입니다.
- `grip(0)`은 닫힘, `grip(100)`은 완전히 열림입니다.
- 이동 명령은 도착할 때까지 그 칸에서 기다립니다.
- 카메라로 집는 칸은 맨 뒤입니다. 그 전에 다른 터미널에서 `python yolo/pose/roi_cloud.py`를 켜 둡니다.

## 1. 팔 켜기

가상 팔입니다. 실기는 이 노트북 맨 끝에서만 다룹니다.

In [ ]:
from motion import Arm, ArmError, load_pick_targets

arm = Arm()
print(arm.where())

`where()` 키입니다.

- `joints_deg`: 관절 6개, 도
- `gripper`: 0이 닫힘, 100이 열림
- `xyz_mm`: 그리퍼 끝 위치
- `rpy_deg`: roll, pitch, yaw

## 2. 시작 자세와 그리퍼

`initial()`은 작업 시작 자세이고 그리퍼는 닫혀 있습니다. `home()`은 관절 0인 L자이고 그리퍼는 반쯤 열립니다.

In [ ]:
arm.initial()
arm.grip(100)
print(arm.where())

## 3. 상대 이동

`"rel"`은 지금 위치에 더합니다. 자세를 적지 않으면 지금 자세를 유지합니다. `speed`는 직선 속도(mm/s)이고, 생략하면 25입니다.

In [ ]:
arm.moveL([30, 0, 0], "rel", speed=10)
arm.moveL([0, 20, 15], "rel")
print(arm.where()["xyz_mm"])

## 4. 절대 위치와 자세

`"abs"`는 베이스 기준 목표입니다. `rpy`는 더하는 각도가 아니라 목표 자세입니다. `path="joint"`이면 그리퍼 끝이 직선으로 가지 않을 수 있습니다.

In [ ]:
arm.moveL([200, 0, 120], "abs")
arm.moveL([180, 40, 90], "abs", rpy=[0, 80, 0])
print(arm.where())

## 5. 지금 높이를 읽어서 올리기

목표 높이를 직접 계산할 때 `where()`로 읽습니다.

In [ ]:
here = arm.where()
x, y, z = here["xyz_mm"]
arm.moveL([x, y, z + 40], "abs")
print(arm.where()["xyz_mm"])

## 6. 관절

`moveJ`는 관절 각을 돌립니다. 그리퍼 끝은 직선이 아닙니다. `"rel"`에서 `gripper`는 도가 아니라 0–100 눈금에 더합니다.

In [ ]:
arm.moveJ([0, -40, 40, 0, -90, 0], "abs")
arm.moveJ([0, 10, 0, 0, 0, 0], "rel", gripper=20)
print(arm.where())

## 7. 홈으로 돌아가기

In [ ]:
arm.home()
print(arm.where())

## 8. 좌표를 적어서 집기

카메라는 쓰지 않습니다. 숫자는 순서를 보여 주는 연습값입니다. 책상 위 브라켓 위치가 아닙니다.

순서: 시작 자세 → 열기 → 내리기 → 닫기 → 들기 → 옆으로 → 내리기 → 열기 → 빠지기.

In [ ]:
arm.initial()
arm.grip(100)
arm.moveL([0, 0, -40], "rel")
arm.grip(0)
arm.moveL([0, 0, 50], "rel")
arm.moveL([0, 80, 0], "rel")
arm.moveL([0, 0, -40], "rel")
arm.grip(100)
arm.moveL([0, 0, 40], "rel")
arm.home()

## 9. 카메라로 집기

이 칸만 카메라가 필요합니다.

1. 다른 터미널에서 `python yolo/pose/roi_cloud.py`를 켜 둡니다. CAD 맞춤과 로봇 기준 좌표는 그 창이 처음부터 합니다.
2. 브라켓이 화면에 보여야 합니다.
3. 펜던트의 물체 위치 버튼과 이 칸을 동시에 누르지 않습니다. 등록 요청이 겹칩니다.

`load_pick_targets(arm)`이 대기 위치 `p_xyz`, 집기 위치 `g_xyz`, 놓기 자세 `place_rpy`를 채웁니다. 내려놓는 X, Y는 여기서 정합니다. 창이 없으면 `ArmError`가 나고 이 칸은 집기를 건너뜁니다.

In [ ]:
drop_x, drop_y = 150.0, -100.0

try:
    pick = load_pick_targets(arm)
except ArmError as exc:
    print(exc)
else:
    z_g = pick.g_xyz[2]
    arm.initial()
    arm.moveL(pick.p_xyz, "abs", rpy=pick.p_rpy)
    arm.grip(30)
    arm.moveL(pick.g_xyz, "abs", rpy=pick.g_rpy)
    arm.grip(0)
    arm.moveL([0, 0, 30], "rel")
    arm.moveL([drop_x, drop_y, z_g + 50], "abs", rpy=pick.place_rpy, path="joint")
    arm.moveL([drop_x, drop_y, z_g], "abs", rpy=pick.place_rpy)
    arm.grip(30)
    arm.moveL([0, 0, 50], "rel")
    arm.initial()

## 10. 수업 중에 자주 나오는 실패

아래는 일부러 실패시켜서 메시지를 보여 주는 칸입니다. 팔은 그대로 둡니다.

- `moveL`의 두 번째 인자를 빼면 `TypeError`입니다. `"abs"` 또는 `"rel"`이 필요합니다.
- 좌표를 `[30, 0, 0]`처럼 리스트로 넘기지 않으면 mode가 어긋납니다.
- 단위는 mm입니다. `0.2`는 0.2m가 아니라 0.2mm입니다.
- `rpy`는 지금 자세에 더하는 값이 아닙니다.
- 팔이 닿지 않는 곳은 `ArmError`입니다.

In [ ]:
try:
    arm.moveL([30, 0, 0])
except TypeError as exc:
    print("mode 생략:", exc)

try:
    arm.moveL([800, 0, 100], "abs")
except ArmError as exc:
    print("닿지 않는 곳:", exc)

## 11. 끄기

`close()` 다음에 `where()`를 부르면 `이미 종료된 Arm`이 납니다. 다시 움직이려면 커널을 재시작하고 1번 칸부터 합니다.

In [ ]:
arm.close()

## 12. 실기 (강사 시연만)

학생 자리에서는 실행하지 않습니다. 팔이 포트에 연결되어 있고, 위의 가상 팔을 `close()`한 뒤입니다. `REAL = True`로 바꾼 다음 이 칸만 실행합니다. 연결이 안 되면 `실기 연결 실패`가 납니다.

In [ ]:
REAL = False

if REAL:
    arm = Arm(mode="real")
    print(arm.where())
    arm.initial()
    arm.close()
else:
    print("실기 시연이 아닙니다. REAL = True 로 바꾼 뒤 이 칸만 실행합니다.")